# 02 — Data Preprocessing & Feature Engineering
Clean invalid sensor values, preserve sensor availability, engineer features, impute missing values, and build the model-ready `X` / `y_failure` / `y_rul` datasets. The reusable implementation lives in `src/data_pipeline.py` so training and inference never drift apart.

In [ ]:
import sys
sys.path.append('..')
from src.data_pipeline import load_raw_data, clean_and_engineer, build_datasets

In [ ]:
df = clean_and_engineer(load_raw_data())
X, y_failure, y_rul = build_datasets(df)

print('Cleaned dataset:', df.shape)
print('Features:', X.shape)
print('Failure target:', y_failure.shape, '| positive rate: %.2f%%' % (y_failure.mean() * 100))
print('RUL target:', y_rul.shape)
print('Remaining missing values in X:', X.isna().sum().sum())

### Engineered features
On top of the original sensor columns, `clean_and_engineer` adds domain-driven features: `Machine_Age`, `Maintenance_Overdue_Ratio`, `Failure_History_Rate`, `Maintenance_Frequency`, `Sensor_Stress_Index`, `Vibration_Power_Ratio`, `Fluid_Level_Deficit`, `Error_Rate_30d`, `AI_Override_Rate`, `Age_Overdue_Interaction`, and `Hours_Per_Maintenance`, plus a `<sensor>_Available` flag for each partially-fitted sensor.

In [ ]:
engineered_cols = [
    'Machine_Age', 'Maintenance_Overdue_Ratio', 'Failure_History_Rate', 'Maintenance_Frequency',
    'Sensor_Stress_Index', 'Vibration_Power_Ratio', 'Fluid_Level_Deficit', 'Error_Rate_30d',
    'AI_Override_Rate', 'Age_Overdue_Interaction', 'Hours_Per_Maintenance',
]
X[engineered_cols].describe().T

In [ ]:
from sklearn.model_selection import train_test_split

# Illustrative 80/20 split for exploration. src/train_models.py uses a stricter
# 60/20/20 train/validation/test split so the validation set can be used for
# threshold tuning without touching the held-out test set.
X_train, X_test, y_failure_train, y_failure_test, y_rul_train, y_rul_test = train_test_split(
    X, y_failure, y_rul, test_size=0.20, random_state=42, stratify=y_failure,
)
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)